# 🍽️ SoulNutri - Treinamento YOLOv8

**Instruções:**
1. Vá em `Runtime > Change runtime type > GPU (T4)`
2. Clique em `Runtime > Run all`
3. Quando pedir, faça upload do arquivo `dataset.zip`
4. Aguarde ~1-2 horas
5. Baixe o modelo treinado no final

---

In [ ]:
#@title 1️⃣ Verificar GPU
!nvidia-smi
import torch
print(f"\n✅ GPU disponível: {torch.cuda.is_available()}")
print(f"   Dispositivo: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
#@title 2️⃣ Instalar YOLOv8
!pip install ultralytics -q
from ultralytics import YOLO
print("✅ Ultralytics instalado!")

In [ ]:
#@title 3️⃣ Upload do Dataset
from google.colab import files
import zipfile
import os

print("📁 Faça upload do arquivo 'dataset.zip'...")
uploaded = files.upload()

# Extrair
for filename in uploaded.keys():
    print(f"\n📦 Extraindo {filename}...")
    with zipfile.ZipFile(filename, 'r') as zip_ref:
        zip_ref.extractall('/content/dataset')
    os.remove(filename)

# Verificar estrutura
!ls -la /content/dataset/
print("\n✅ Dataset extraído!")

In [ ]:
#@title 4️⃣ Verificar Dataset
import os

# Encontrar diretório correto
dataset_path = '/content/dataset'
if os.path.exists('/content/dataset/datasets_augmented'):
    dataset_path = '/content/dataset/datasets_augmented'
elif os.path.exists('/content/dataset/train'):
    dataset_path = '/content/dataset'

train_path = os.path.join(dataset_path, 'train')
val_path = os.path.join(dataset_path, 'val')

# Contar
train_classes = len([d for d in os.listdir(train_path) if os.path.isdir(os.path.join(train_path, d))])
train_images = sum(len(files) for _, _, files in os.walk(train_path))
val_images = sum(len(files) for _, _, files in os.walk(val_path))

print(f"📊 Dataset encontrado em: {dataset_path}")
print(f"   Classes: {train_classes}")
print(f"   Imagens treino: {train_images}")
print(f"   Imagens validação: {val_images}")
print(f"\n✅ Dataset OK!")

In [ ]:
#@title 5️⃣ Treinar Modelo 🚀
#@markdown **Configurações de Treinamento**
epochs = 100  #@param {type:"integer"}
batch_size = 32  #@param {type:"integer"}
image_size = 224  #@param {type:"integer"}

from ultralytics import YOLO

# Carregar modelo pré-treinado (nano - mais rápido)
model = YOLO('yolov8n-cls.pt')

# Treinar
print(f"\n🚀 Iniciando treinamento...")
print(f"   Epochs: {epochs}")
print(f"   Batch: {batch_size}")
print(f"   Imagem: {image_size}x{image_size}")
print(f"\n⏳ Isso vai levar ~1-2 horas. Relaxe!\n")

results = model.train(
    data=dataset_path,
    epochs=epochs,
    imgsz=image_size,
    batch=batch_size,
    patience=20,  # Early stopping
    device=0,  # GPU
    workers=2,
    project='soulnutri',
    name='food_classifier',
    exist_ok=True,
    pretrained=True,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=15,
    translate=0.1,
    scale=0.5,
    flipud=0.0,
    fliplr=0.5,
)

print("\n🎉 Treinamento concluído!")

In [ ]:
#@title 6️⃣ Avaliar Modelo
from ultralytics import YOLO

# Carregar melhor modelo
best_model = YOLO('/content/soulnutri/food_classifier/weights/best.pt')

# Validar
metrics = best_model.val()

print(f"\n📊 Resultados:")
print(f"   Acurácia Top-1: {metrics.top1:.2%}")
print(f"   Acurácia Top-5: {metrics.top5:.2%}")

In [ ]:
#@title 7️⃣ Testar com Imagem
from google.colab import files
from ultralytics import YOLO
from PIL import Image
import matplotlib.pyplot as plt

print("📸 Faça upload de uma imagem de comida para testar:")
uploaded = files.upload()

for filename in uploaded.keys():
    # Carregar modelo
    model = YOLO('/content/soulnutri/food_classifier/weights/best.pt')
    
    # Predição
    results = model.predict(filename, verbose=False)
    
    # Mostrar resultado
    img = Image.open(filename)
    plt.figure(figsize=(8, 8))
    plt.imshow(img)
    plt.axis('off')
    
    # Top 3 predições
    probs = results[0].probs
    top3_idx = probs.top5[:3]
    top3_conf = probs.top5conf[:3]
    names = results[0].names
    
    title = "\n".join([f"{names[idx]}: {conf:.1%}" for idx, conf in zip(top3_idx, top3_conf)])
    plt.title(f"🍽️ Identificação:\n{title}", fontsize=14)
    plt.show()
    
    print(f"\n✅ Top predição: {names[top3_idx[0]]} ({top3_conf[0]:.1%})")

In [ ]:
#@title 8️⃣ Exportar e Baixar Modelo
from ultralytics import YOLO
from google.colab import files
import shutil

model = YOLO('/content/soulnutri/food_classifier/weights/best.pt')

# Exportar para ONNX (compatível com mais plataformas)
print("📦 Exportando para ONNX...")
model.export(format='onnx', imgsz=224, simplify=True)

# Criar ZIP com os modelos
print("\n📁 Criando pacote de modelos...")
shutil.make_archive(
    '/content/soulnutri_model',
    'zip',
    '/content/soulnutri/food_classifier/weights'
)

print("\n⬇️ Baixando modelo...")
files.download('/content/soulnutri_model.zip')

print("\n✅ Download iniciado!")
print("\n📌 O arquivo contém:")
print("   - best.pt (PyTorch - para Python)")
print("   - best.onnx (ONNX - universal)")

---
## 🎉 Pronto!

Seu modelo foi treinado e baixado. Agora:

1. Extraia o `soulnutri_model.zip`
2. Coloque `best.pt` em `/app/ml/models/`
3. O SoulNutri usará automaticamente o novo modelo

---
*SoulNutri - Seu agente de nutrição virtual*